In [1]:
# Install required packages for pure PyTorch implementation
# Run this cell if you haven't installed the required packages

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install scikit-learn pillow matplotlib numpy
# !pip install scikit-image tqdm

print("Required packages for this notebook:")
print("- torch, torchvision, torchaudio")
print("- scikit-learn")
print("- pillow (PIL)")
print("- matplotlib")
print("- numpy")
print("- scikit-image")
print("- tqdm (for progress bars)")
print("- pandas (optional)")
print("\nTo install, uncomment and run the pip install commands above.")

Required packages for this notebook:
- torch, torchvision, torchaudio
- scikit-learn
- pillow (PIL)
- matplotlib
- numpy
- scikit-image
- tqdm (for progress bars)
- pandas (optional)

To install, uncomment and run the pip install commands above.


In [2]:
# Install tqdm for progress bars
try:
    from tqdm.auto import tqdm
    print("✅ tqdm already installed")
except ImportError:
    print("📦 Installing tqdm...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
    from tqdm.auto import tqdm
    print("✅ tqdm installed successfully!")

# Test tqdm
import time
print("\n🧪 Testing progress bar:")
for i in tqdm(range(5), desc="Testing"):
    time.sleep(0.1)
print("🎉 Progress bar working correctly!")

✅ tqdm already installed

🧪 Testing progress bar:


Testing:   0%|          | 0/5 [00:00<?, ?it/s]

🎉 Progress bar working correctly!


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
print("GPU available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name())

GPU available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: NVIDIA GeForce RTX 5070 Ti


In [4]:
# Thiết lập device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Hyperparameters
monitor='val_accuracy'
epochs=40
batch_size=16
# Vision Transformer (ViT) models are typically trained on 224x224 images.
input_shape=(224, 224, 3)
Verbose=True
learning_rate=0.001
num_classes=15

This source code requires a **HIGH RAM** machine.

You might need to install this on your system:

apt-get install python3-opencv git

For PyTorch: pip install torch torchvision torchaudio

In [6]:
import os

# if not os.path.isdir('k'):
#   !git clone https://github.com/joaopauloschuler/k-neural-api.git k
# else:
#   !cd k && git pull

# !cd k && pip install .

In [7]:
import sys
print("Python version")
print (sys.version)
print("Version info.")
print (sys.version_info)
import importlib
import skimage
print('skimage version',  skimage.__version__)
import torch
import torchvision
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)

# Pure PyTorch implementation - no CAI dependencies
print("Using pure PyTorch implementation!")

Python version
3.10.11 | packaged by Anaconda, Inc. | (main, May 16 2023, 00:55:32) [MSC v.1916 64 bit (AMD64)]
Version info.
sys.version_info(major=3, minor=10, micro=11, releaselevel='final', serial=0)
skimage version 0.25.2
PyTorch version: 2.7.1+cu128
Torchvision version: 0.22.1+cu128
Using pure PyTorch implementation!


In [8]:
# url_zip_file="https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/d5652a28-c1d8-4b76-97f3-72fb80f94efc/file_downloaded"
# local_zip_file="plant_leaf.zip"
# expected_folder_name="plant_leaf"
# cai.datasets.download_zip_and_extract(
#     url_zip_file=url_zip_file, local_zip_file=local_zip_file, 
#     expected_folder_name=expected_folder_name, Verbose=Verbose)

In [9]:
import random
import os
import multiprocessing
import glob
import numpy as np
from sklearn.model_selection import train_test_split
import sklearn.utils.class_weight
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
%matplotlib inline

In [10]:
# !rm -r plant_leaf/Plant_leave_diseases_dataset_without_augmentation/Background_without_leaves -R
data_dir = "dataset/PlantVillage"
print(os.listdir(data_dir))

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_healthy', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_mosaic_virus', 'Tomato__Tomato_YellowLeaf__Curl_Virus']


In [11]:
# Pure PyTorch data loading function - MEMORY EFFICIENT
def get_image_paths_and_labels(root_dir, verbose=True):
    """
    Scans the directory to get a list of image paths and their corresponding labels.
    Does NOT load images into memory.
    """
    image_paths = []
    labels = []
    classes = []
    class_to_idx = {}
    
    if verbose:
        print(f"Scanning for images in: {root_dir}")
        
    for class_name in sorted(os.listdir(root_dir)):
        class_path = os.path.join(root_dir, class_name)
        if os.path.isdir(class_path):
            if class_name not in class_to_idx:
                classes.append(class_name)
                class_to_idx[class_name] = len(classes) - 1
            
            class_idx = class_to_idx[class_name]
            
            num_images = 0
            for img_file in os.listdir(class_path):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(class_path, img_file))
                    labels.append(class_idx)
                    num_images += 1
            
            if verbose:
                print(f"  Found {num_images} images for class: {class_name}")

    if verbose:
        print(f"\nTotal images found: {len(image_paths)}")
        print(f"Total classes found: {len(classes)}")
        
    return image_paths, labels, classes

def split_data(image_paths, labels, seed=7, training_size=0.6, validation_size=0.2, test_size=0.2, verbose=True):
    """
    Splits the lists of paths and labels into training, validation, and test sets.
    """
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelBinarizer
    import sklearn.utils.class_weight

    # Ensure the splits add up to 1
    assert training_size + validation_size + test_size == 1.0

    # Convert labels to one-hot for stratified splitting
    lb = LabelBinarizer()
    labels_onehot = lb.fit_transform(labels)

    # First split: train and temp (val + test)
    temp_size = validation_size + test_size
    train_x_paths, temp_x_paths, train_y, temp_y = train_test_split(
        image_paths, labels_onehot, test_size=temp_size, random_state=seed, stratify=labels
    )

    # Second split: validation and test
    val_ratio = validation_size / temp_size
    val_x_paths, test_x_paths, val_y, test_y = train_test_split(
        temp_x_paths, temp_y, test_size=(1 - val_ratio), random_state=seed, stratify=np.argmax(temp_y, axis=1)
    )

    # Calculate class weights for handling imbalanced data
    class_weights = sklearn.utils.class_weight.compute_class_weight(
        'balanced', classes=np.unique(labels), y=labels
    )
    classweight = dict(enumerate(class_weights))

    if verbose:
        print(f"\nData split:")
        print(f"  Training set:   {len(train_x_paths)} samples")
        print(f"  Validation set: {len(val_x_paths)} samples")
        print(f"  Test set:       {len(test_x_paths)} samples")
        print(f"Class weights calculated.")

    return train_x_paths, val_x_paths, test_x_paths, train_y, val_y, test_y, classweight

# Get paths and labels without loading images
all_image_paths, all_labels, classes = get_image_paths_and_labels(root_dir=data_dir, verbose=Verbose)

# Split the data
train_x, val_x, test_x, train_y, val_y, test_y, classweight = split_data(
    all_image_paths, 
    all_labels,
    seed=7,
    training_size=0.6, 
    validation_size=0.2, 
    test_size=0.2,
    verbose=Verbose
)

Scanning for images in: dataset/PlantVillage
  Found 997 images for class: Pepper__bell___Bacterial_spot
  Found 1478 images for class: Pepper__bell___healthy
  Found 1000 images for class: Potato___Early_blight
  Found 1000 images for class: Potato___Late_blight
  Found 152 images for class: Potato___healthy
  Found 2127 images for class: Tomato_Bacterial_spot
  Found 1000 images for class: Tomato_Early_blight
  Found 1909 images for class: Tomato_Late_blight
  Found 952 images for class: Tomato_Leaf_Mold
  Found 1771 images for class: Tomato_Septoria_leaf_spot
  Found 1676 images for class: Tomato_Spider_mites_Two_spotted_spider_mite
  Found 1404 images for class: Tomato__Target_Spot
  Found 3208 images for class: Tomato__Tomato_YellowLeaf__Curl_Virus
  Found 373 images for class: Tomato__Tomato_mosaic_virus
  Found 1591 images for class: Tomato_healthy

Total images found: 20638
Total classes found: 15

Data split:
  Training set:   12382 samples
  Validation set: 4128 samples
  Tes

In [12]:
print(classes)

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [13]:
# Since train_x, val_x, and test_x are now lists of file paths, we use len() to see the number of samples.
print(f"Number of training samples: {len(train_x)}")
print(f"Number of validation samples: {len(val_x)}")
print(f"Number of test samples: {len(test_x)}")

# The labels are still numpy arrays, so .shape is correct here.
print(f"\nShape of training labels: {train_y.shape}")
print(f"Shape of validation labels: {val_y.shape}")
print(f"Shape of test labels: {test_y.shape}")


Number of training samples: 12382
Number of validation samples: 4128
Number of test samples: 4128

Shape of training labels: (12382, 15)
Shape of validation labels: (4128, 15)
Shape of test labels: (4128, 15)


In [14]:
# Custom Dataset class for PyTorch - MEMORY EFFICIENT
class PlantDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        """
        Args:
            image_paths (list of str): List of paths to the images.
            labels (numpy.ndarray): One-hot encoded labels.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image from path
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        # Get label and convert from one-hot to class index
        label = np.argmax(self.labels[idx])
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [15]:
# Data transforms - updated for ViT standard (224x224)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = PlantDataset(train_x, train_y, transform=train_transform)
val_dataset = PlantDataset(val_x, val_y, transform=val_test_transform)
test_dataset = PlantDataset(test_x, test_y, transform=val_test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


🚀 Using 32 CPU workers for data loading.


In [16]:
# Debug: Check data consistency
print("🔍 Data Consistency Check:")
print(f"Input shape setting: {input_shape}")
print(f"Target size for loading: ({input_shape[0]}, {input_shape[1]})")

# Check a few samples from the dataset
for i in range(2):
    image, label = train_dataset[i]
    print(f"\nSample {i+1}:")
    print(f"  Image tensor shape: {image.shape}")
    print(f"  Image tensor range: [{image.min():.3f}, {image.max():.3f}]")
    print(f"  Label: {label} -> {classes[label]}")

# Check if data matches expected range after transforms
print(f"\n📊 After transforms:")
print(f"  Expected range: ImageNet normalized (~ [-2, 2])")
print(f"  Actual range: [{image.min():.3f}, {image.max():.3f}]")

print(f"\n✅ Data loading completed. Ready for training with:")
print(f"  - Image size: 160x160 (for ViT, for fair comparison)")
print(f"  - Batch size: {batch_size}")
print(f"  - Epochs: {epochs}")
print(f"  - Classes: {len(classes)}")

🔍 Data Consistency Check:
Input shape setting: (224, 224, 3)
Target size for loading: (224, 224)

Sample 1:
  Image tensor shape: torch.Size([3, 224, 224])
  Image tensor range: [-2.118, 1.071]
  Label: 14 -> Tomato_healthy

Sample 2:
  Image tensor shape: torch.Size([3, 224, 224])
  Image tensor range: [-1.998, 0.496]
  Label: 7 -> Tomato_Late_blight

📊 After transforms:
  Expected range: ImageNet normalized (~ [-2, 2])
  Actual range: [-1.998, 0.496]

✅ Data loading completed. Ready for training with:
  - Image size: 160x160 (for ViT, for fair comparison)
  - Batch size: 16
  - Epochs: 40
  - Classes: 15


In [17]:
# Vision Transformer model definition - SIMPLIFIED
class ViTPlantClassifier(nn.Module):
    def __init__(self, num_classes=15, pretrained=True):
        super(ViTPlantClassifier, self).__init__()
        # Load the pretrained Vision Transformer model
        # The 'weights' parameter is the modern way to load pretrained models
        self.backbone = vit_b_16(weights='IMAGENET1K_V1' if pretrained else None)
        
        # Replace the original classifier head with a new one for our number of classes
        in_features = self.backbone.heads.head.in_features
        self.backbone.heads.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # The forward pass is now simple, as we are using the standard image size
        return self.backbone(x)

# Function to save PyTorch model
def save_pytorch_model(model, model_dir):
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, 'model.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_class': model.__class__.__name__
    }, model_path)
    print(f"Model saved to {model_path}")

# Function to load PyTorch model
def load_pytorch_model(model_dir, model_class, num_classes=15):
    model_path = os.path.join(model_dir, 'model.pth')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        model = model_class(num_classes=num_classes)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model
    else:
        raise FileNotFoundError(f"No model found at {model_path}")

In [18]:
# Debug: Check ViT architecture
print("🔍 Checking Vision Transformer Architecture:")

# Test model creation
test_model = vit_b_16(pretrained=True)
print(f"Original ViT classifier (head):")
print(test_model.heads)

# Get the correct input dimension
in_features = test_model.heads.head.in_features
print(f"\nInput dimension for classifier should be: {in_features}")

del test_model

🔍 Checking Vision Transformer Architecture:


c:\Users\PC\anaconda3\envs\cuda\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\PC\anaconda3\envs\cuda\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Original ViT classifier (head):
Sequential(
  (head): Linear(in_features=768, out_features=1000, bias=True)
)

Input dimension for classifier should be: 768


In [19]:
# Safe model creation function using the simplified ViTPlantClassifier class
def create_vit_model(num_classes=15, pretrained=True):
    """
    Creates a Vision Transformer model by instantiating the ViTPlantClassifier class.
    """
    print("Creating model using simplified ViTPlantClassifier class...")
    # We no longer need to pass image_size as it's handled by the standard model
    return ViTPlantClassifier(num_classes=num_classes, pretrained=pretrained)

# Test the safe model creation
print("🧪 Testing safe model creation:")
test_model = create_vit_model(num_classes=15, pretrained=True)

# Test forward pass with the correct image size
dummy_input = torch.randn(2, 3, 224, 224)  # batch_size=2, image_size=224x224
try:
    with torch.no_grad():
        output = test_model(dummy_input)
    print(f"✅ Forward pass successful! Output shape: {output.shape}")
    print(f"Expected shape: [2, 15] -> Got: {list(output.shape)}")
except Exception as e:
    print(f"❌ Forward pass failed: {e}")

del test_model, dummy_input

🧪 Testing safe model creation:
Creating model using simplified ViTPlantClassifier class...
✅ Forward pass successful! Output shape: torch.Size([2, 15])
Expected shape: [2, 15] -> Got: [2, 15]
✅ Forward pass successful! Output shape: torch.Size([2, 15])
Expected shape: [2, 15] -> Got: [2, 15]


In [20]:
# Training function with improved logging and progress bars
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, device, save_path):
    best_val_acc = 0.0
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []
    
    print(f"🚀 Starting training for {epochs} epochs...")
    print(f"📱 Device: {device}")
    print(f"💾 Model will be saved to: {save_path}")
    print("=" * 80)
    
    # Main epoch progress bar
    epoch_pbar = tqdm(range(epochs), desc="🔄 Training Progress", unit="epoch")
    
    for epoch in epoch_pbar:
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        # Training batch progress bar
        train_pbar = tqdm(train_loader, 
                         desc=f"📚 Epoch {epoch+1}/{epochs} - Training", 
                         leave=False, 
                         unit="batch")
        
        for batch_idx, (data, target) in enumerate(train_pbar):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            # Update training progress bar
            current_acc = 100. * correct / total
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{current_acc:.2f}%'
            })
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        # Validation batch progress bar
        val_pbar = tqdm(val_loader, 
                       desc=f"🔍 Epoch {epoch+1}/{epochs} - Validation", 
                       leave=False, 
                       unit="batch")
        
        with torch.no_grad():
            for data, target in val_pbar:
                data, target = data.to(device), target.to(device)
                output = model(data)
                val_loss += criterion(output, target).item()
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
                
                # Update validation progress bar
                current_acc = 100. * correct / total if total > 0 else 0
                val_pbar.set_postfix({
                    'Loss': f'{criterion(output, target).item():.4f}',
                    'Acc': f'{current_acc:.2f}%'
                })
        
        val_loss /= len(val_loader)
        val_acc = 100. * correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        # Save best model
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'classes': classes,
                'train_losses': train_losses,
                'train_accs': train_accs,
                'val_losses': val_losses,
                'val_accs': val_accs
            }, save_path)
        
        scheduler.step()
        
        # Update main epoch progress bar
        epoch_pbar.set_postfix({
            'Train_Loss': f'{train_loss:.4f}',
            'Train_Acc': f'{train_acc:.2f}%',
            'Val_Loss': f'{val_loss:.4f}',
            'Val_Acc': f'{val_acc:.2f}%',
            'Best_Val': f'{best_val_acc:.2f}%',
            'Best': '⭐' if is_best else ''
        })
        
        # Print detailed epoch summary
        status_icon = "🆕" if is_best else "📊"
        print(f'\n{status_icon} Epoch [{epoch+1}/{epochs}] Summary:')
        print(f'   🏋️  Training   - Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%')
        print(f'   🎯 Validation - Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%')
        if is_best:
            print(f'   ⭐ New best model saved! Val Accuracy: {val_acc:.2f}%')
        print(f'   📈 Best so far: {best_val_acc:.2f}%')
        print('-' * 80)
    
    epoch_pbar.close()
    print(f"\n🎉 Training completed successfully!")
    print(f"🏆 Best validation accuracy achieved: {best_val_acc:.2f}%")
    print(f"💾 Best model saved to: {save_path}")
    print("=" * 80)
    
    return train_losses, train_accs, val_losses, val_accs, best_val_acc

In [21]:
# Function to plot training history
def plot_training_history(train_losses, train_accs, val_losses, val_accs):
    """
    Plot training and validation loss and accuracy curves
    """
    epochs_range = range(len(train_losses))
    
    plt.figure(figsize=(15, 5))
    
    # Plot loss
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, train_losses, label='Training Loss', color='blue')
    plt.plot(epochs_range, val_losses, label='Validation Loss', color='red')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Plot accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accs, label='Training Accuracy', color='blue')
    plt.plot(epochs_range, val_accs, label='Validation Accuracy', color='red')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    # Plot learning rate (if using scheduler)
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, [0.001 * (0.1 ** (epoch // 10)) for epoch in epochs_range], 
             label='Learning Rate', color='green')
    plt.title('Learning Rate Schedule')
    plt.xlabel('Epochs')
    plt.ylabel('Learning Rate')
    plt.legend()
    plt.grid(True)
    plt.yscale('log')
    
    plt.tight_layout()
    plt.show()

# Function to plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, classes, normalize=False):
    """
    Plot confusion matrix
    """
    from sklearn.metrics import confusion_matrix
    import itertools
    
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')
    
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                horizontalalignment="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

In [22]:
# Enhanced evaluation function with progress bars
def evaluate_model(model, test_loader, criterion, device, classes):
    """
    Evaluate model with progress bar and detailed metrics
    """
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_targets = []
    class_correct = list(0. for i in range(len(classes)))
    class_total = list(0. for i in range(len(classes)))
    
    print("🔍 Evaluating model...")
    
    # Progress bar for evaluation
    eval_pbar = tqdm(test_loader, desc="📊 Testing", unit="batch")
    
    with torch.no_grad():
        for data, target in eval_pbar:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            test_loss += loss.item()
            
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            # Collect all predictions and targets
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            
            # Per-class accuracy calculation
            c = (predicted == target).squeeze()
            for i in range(target.size(0)):
                label = target[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1
            
            # Update progress bar
            current_acc = 100. * correct / total
            eval_pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{current_acc:.2f}%'
            })
    
    eval_pbar.close()
    
    # Calculate final metrics
    test_loss /= len(test_loader)
    test_acc = 100. * correct / total
    
    print(f"\n📈 Overall Test Results:")
    print(f"   🎯 Test Loss: {test_loss:.4f}")
    print(f"   📊 Test Accuracy: {test_acc:.2f}% ({correct}/{total})")
    
    # Per-class accuracy
    print(f"\n📋 Per-Class Accuracy:")
    for i, class_name in enumerate(classes):
        if class_total[i] > 0:
            class_acc = 100. * class_correct[i] / class_total[i]
            print(f"   {class_name:25s}: {class_acc:6.2f}% ({int(class_correct[i])}/{int(class_total[i])})")
        else:
            print(f"   {class_name:25s}: No samples")
    
    return test_loss, test_acc, np.array(all_predictions), np.array(all_targets)

# Function to run complete evaluation with visualizations
def complete_evaluation(model, test_loader, criterion, device, classes):
    """
    Run complete evaluation with progress bars and visualizations
    """
    print("🚀 Starting Complete Model Evaluation")
    print("=" * 60)
    
    # Evaluate model
    test_loss, test_acc, y_pred, y_true = evaluate_model(model, test_loader, criterion, device, classes)
    
    # Classification report
    print(f"\n📊 Detailed Classification Report:")
    print("=" * 60)
    from sklearn.metrics import classification_report
    print(classification_report(y_true, y_pred, target_names=classes))
    
    # Confusion matrices
    print(f"\n📊 Confusion Matrix Visualization:")
    plot_confusion_matrix(y_true, y_pred, classes, normalize=False)
    plot_confusion_matrix(y_true, y_pred, classes, normalize=True)
    
    return test_loss, test_acc, y_pred, y_true

In [ ]:
# Pure PyTorch training loop for ViT
import torch
import torch.nn as nn
import torch.optim as optim

os.makedirs("modelsCP", exist_ok=True)

for lr in [0.001]:
    basefilename = f'vit-b16-lr{lr}'
    model_dir = os.path.join("./modelsCP", basefilename)
    best_result_file_name = os.path.join(model_dir, 'best_result.pth')
    
    print(f'🚀 Running: {basefilename}')
    print('='*60)
    
    try:
        # Try to load existing model
        if os.path.exists(best_result_file_name):
            print(f"Found existing model at {best_result_file_name}")
            checkpoint = torch.load(best_result_file_name, map_location=device)
            model = create_vit_model(num_classes=num_classes, pretrained=True)
            model.load_state_dict(checkpoint['model_state_dict'])
            model.to(device)
            print(f"✓ Loaded model from {model_dir}")
            print(f"Previous best val accuracy: {checkpoint.get('val_acc', 'Unknown'):.2f}%")
        else:
            raise FileNotFoundError("No existing model found")
    except Exception as e:
        print("Creating new model...")
        model = create_vit_model(num_classes=num_classes, pretrained=True)
        model.to(device)
        print(f"✓ Created new Vision Transformer model")
    
    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr*0.1, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
    
    print(f"📊 Training Configuration:")
    print(f"   - Model: Vision Transformer (ViT-B/16)")
    print(f"   - Learning Rate: {lr}")
    print(f"   - Epochs: {epochs}")
    print(f"   - Batch Size: {batch_size}")
    print(f"   - Device: {device}")
    print('='*60)
    
    # Train the model
    train_losses, train_accs, val_losses, val_accs, best_val_acc = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler, 
        epochs, device, best_result_file_name
    )
    
    print(f'📈 Testing Last Model: {basefilename}')
    # Test the current model
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    test_acc = 100. * correct / total
    test_loss /= len(test_loader)
    print(f'Last Model Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')
    
    print(f'🏆 Best Model Results: {basefilename}')
    # Load and test the best model
    if os.path.exists(best_result_file_name):
        checkpoint = torch.load(best_result_file_name, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        correct = 0
        total = 0
        test_loss = 0.0
        
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                test_loss += criterion(output, target).item()
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
        
        test_acc = 100. * correct / total
        test_loss /= len(test_loader)
        print(f'✨ Best Model Test Loss: {test_loss:.4f}, Best Test Acc: {test_acc:.2f}%')
    
    # Save the final model
    save_pytorch_model(model, model_dir)
    
    print(f'✅ Finished: {basefilename}')
    print('='*70)


# Compare Models

In [ ]:
# Load the best ViT model
print('Vision Transformer Results: ')
model_dir = "./modelsCP/vit-b16-lr0.001"
best_result_file_name = os.path.join(model_dir, 'best_result.pth')

if os.path.exists(best_result_file_name):
    model = create_vit_model(num_classes=num_classes, pretrained=False)
    model.to(device)
    
    checkpoint = torch.load(best_result_file_name, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    test_acc = 100. * correct / total
    test_loss /= len(test_loader)
    print(f'Test Loss: {test_loss:.4f}')
    print(f'Test Accuracy: {test_acc:.2f}%')
else:
    print("Best model not found. Please train the model first.")

In [ ]:
# Enhanced evaluation with progress bars
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

if 'model' in locals():
    print("🔍 Running Enhanced Model Evaluation")
    print("=" * 60)
    
    # Use the new evaluation function with progress bars
    test_loss, test_acc, y_pred_classes, y_true_classes = complete_evaluation(
        model, test_loader, nn.CrossEntropyLoss(), device, classes
    )
    
else:
    print("❌ Model not loaded. Please run the previous cell first.")

In [ ]:
# Visualize training results (run this after training)
if 'train_losses' in locals() and 'val_losses' in locals():
    print("📊 Training History Visualization")
    plot_training_history(train_losses, train_accs, val_losses, val_accs)
    
    print("📈 Training Summary:")
    print(f"Final Training Accuracy: {train_accs[-1]:.2f}%")
    print(f"Final Validation Accuracy: {val_accs[-1]:.2f}%")
    print(f"Best Validation Accuracy: {max(val_accs):.2f}%")
    print(f"Final Training Loss: {train_losses[-1]:.4f}")
    print(f"Final Validation Loss: {val_losses[-1]:.4f}")
else:
    print("Training history not available. Please run the training cell first.")

# Make Prediction

In [ ]:
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torchvision.transforms as transforms

# image_path = "./realImage/late_blight_tomato_leaf5x12001-1.jpg"
image_path = "./realImage/UK_advice-pests-diseases-tomato-leaf-mould_main.jpg"

# Define preprocessing for single image
single_image_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Match ViT training size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if 'model' in locals() and os.path.exists(image_path):
    # Load and preprocess image
    img_rgb = Image.open(image_path).convert('RGB')
    img_tensor = single_image_transform(img_rgb).unsqueeze(0).to(device)  # Add batch dimension
    
    # Make prediction
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        predicted_idx = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_idx].item()
    
    # Display image
    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    plt.title("Original IMG (RGB)")
    plt.axis('off')
    plt.show()
    
    # Display results
    print(f"IMG: {os.path.basename(image_path)}")
    print(f"Real IMG: {os.path.basename(os.path.dirname(image_path))}")
    print(f"Prediction: {classes[predicted_idx]} ({confidence:.4f})")
    
    # Show top 3 predictions
    top3_prob, top3_idx = torch.topk(probabilities[0], 3)
    print("\nTop 3 predictions:")
    for i in range(3):
        idx = top3_idx[i].item()
        prob = top3_prob[i].item()
        print(f"{i+1}. {classes[idx]}: {prob:.4f}")
else:
    if 'model' not in locals():
        print("Model not loaded. Please run the model loading cell first.")
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")